# Stemming vs Lemmatization Project

**NLP Traditional Preprocessing - Text Normalization Comparison**

## Project Overview

Compare stemming (Porter, Snowball) vs lemmatization (spaCy) on text classification. Measure vocabulary reduction and classification quality.

## Learning Objectives

1. Understand stemming vs lemmatization.
2. Implement Porter, Snowball, spaCy normalization.
3. Measure vocabulary and classification impact.
4. Learn when each is appropriate.

## Problem Statement

Which normalization produces better features for NLP classification?

## Why This Project Matters

- Stemming is faster but cruder.
- Lemmatization is linguistically correct.
- Vocabulary reduction improves generalization.
- Wrong choice hurts performance.

## Dataset Overview

IMDB reviews, 4K subset, binary sentiment.

## Dataset Source and License Notes

Hugging Face `imdb`. Free for research.

## Environment Setup

In [ ]:
import subprocess,sys,importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","scikit-learn","nltk","spacy","datasets"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable,"-m","spacy","download","en_core_web_sm"])
import nltk; nltk.download("punkt",quiet=True); nltk.download("punkt_tab",quiet=True); nltk.download("wordnet",quiet=True)
print("Ready.")

## Imports

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns,re,time,warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report,confusion_matrix,ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from nltk.stem import PorterStemmer,SnowballStemmer
import spacy
from datasets import load_dataset
warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
%matplotlib inline
print("Loaded.")

## Configuration / Constants

In [ ]:
SEED=42; N_SAMPLES=4000; TEST_SIZE=0.2; MAX_FEATURES=10000
CLASS_NAMES=["Negative","Positive"]; np.random.seed(SEED)

## Dataset Download or Loading

In [ ]:
ds=load_dataset("imdb",split="train")
df=ds.to_pandas().sample(n=N_SAMPLES,random_state=SEED).reset_index(drop=True)
print(f"Shape:{df.shape}")

## Data Validation Checks

In [ ]:
print("Missing:",df.isnull().sum().to_dict())
df=df.drop_duplicates(subset=["text"]).reset_index(drop=True)
assert df["label"].isin([0,1]).all()
df["text_len"]=df["text"].str.len(); df["word_count"]=df["text"].str.split().apply(len)
print("Validation passed.")

## Exploratory Data Analysis

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(df["text_len"],bins=50,color="steelblue",edgecolor="white"); axes[0].set_title("Text Length")
for l,n in enumerate(CLASS_NAMES): axes[1].hist(df[df["label"]==l]["word_count"],bins=30,alpha=0.5,label=n)
axes[1].set_title("Words by Sentiment"); axes[1].legend()
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
cc=df["label"].value_counts().sort_index()
fig,ax=plt.subplots(figsize=(6,4))
ax.bar(CLASS_NAMES,cc.values,color=["#C44E52","#55A868"]); ax.set_title("Sentiment Distribution")
plt.tight_layout(); plt.show()

## Train/Validation/Test Split Strategy

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df["text"],df["label"],test_size=TEST_SIZE,random_state=SEED,stratify=df["label"])
print(f"Train:{len(X_train)}|Test:{len(X_test)}")

## Preprocessing Strategy - Normalization Functions

None (lowercase only), Porter Stemmer, Snowball Stemmer, spaCy Lemmatizer.

In [ ]:
porter=PorterStemmer(); snowball=SnowballStemmer("english")
nlp=spacy.load("en_core_web_sm",disable=["ner","parser"]); nlp.max_length=100000
def apply_none(text): return " ".join(re.findall(r"\w+",text.lower()))
def apply_porter(text): return " ".join(porter.stem(w) for w in re.findall(r"\w+",text.lower()))
def apply_snowball(text): return " ".join(snowball.stem(w) for w in re.findall(r"\w+",text.lower()))
def apply_spacy_lemma(text):
    doc=nlp(text[:5000])
    return " ".join(t.lemma_.lower() for t in doc if t.is_alpha)
NORMALIZERS={"None (lowercase)": apply_none, "Porter Stemmer": apply_porter,
    "Snowball Stemmer": apply_snowball, "spaCy Lemmatizer": apply_spacy_lemma}

### Example comparison

In [ ]:
test_words=["running","better","studies","wolves","happily","caring","flies"]
print(f"{'Word':<12}{'Porter':<12}{'Snowball':<12}{'spaCy':<15}")
for w in test_words:
    doc=nlp(w)
    print(f"{w:<12}{porter.stem(w):<12}{snowball.stem(w):<12}{doc[0].lemma_ if len(doc)>0 else w:<15}")
sample="The running dogs were better than the swimming cats yesterday"
for n,fn in NORMALIZERS.items(): print(f"{n:25s}: {fn(sample)}")

## Feature Engineering - Vocabulary Impact

In [ ]:
vs=[]
for n,fn in NORMALIZERS.items():
    t0=time.time(); tn=X_train.apply(fn); el=time.time()-t0
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2))
    X_tr=tfidf.fit_transform(tn)
    vs.append({"Method":n,"Vocab":len(tfidf.vocabulary_),"Time(s)":round(el,2)})
print(pd.DataFrame(vs).to_string(index=False))

In [ ]:
vdf=pd.DataFrame(vs)
fig,axes=plt.subplots(1,2,figsize=(12,4))
colors=["#4C72B0","#DD8452","#55A868","#C44E52"]
axes[0].bar(vdf["Method"],vdf["Vocab"],color=colors); axes[0].set_title("Vocab Size"); axes[0].tick_params(axis="x",rotation=25)
axes[1].bar(vdf["Method"],vdf["Time(s)"],color=colors); axes[1].set_title("Time(s)"); axes[1].tick_params(axis="x",rotation=25)
plt.tight_layout(); plt.show()

## Baseline Model

In [ ]:
Xtn=X_train.apply(apply_none); Xten=X_test.apply(apply_none)
tfidf_b=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2))
X_tr_b=tfidf_b.fit_transform(Xtn); X_te_b=tfidf_b.transform(Xten)
bl=MultinomialNB(); bl.fit(X_tr_b,y_train); y_pb=bl.predict(X_te_b)
bl_acc=accuracy_score(y_test,y_pb); bl_f1=f1_score(y_test,y_pb)
print(f"BASELINE: Acc={bl_acc:.4f}, F1={bl_f1:.4f}")
print(classification_report(y_test,y_pb,target_names=CLASS_NAMES))

## Full Normalization x Classifier Comparison

In [ ]:
classifiers={"MultinomialNB":MultinomialNB(),"LogisticRegression":LogisticRegression(max_iter=1000,random_state=SEED),"LinearSVC":LinearSVC(max_iter=2000,random_state=SEED)}
results=[]
for nn,nf in NORMALIZERS.items():
    print(f"Processing: {nn}...")
    Xtrn=X_train.apply(nf); Xten=X_test.apply(nf)
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2))
    X_tr=tfidf.fit_transform(Xtrn); X_te=tfidf.transform(Xten)
    for cn,clf in classifiers.items():
        m=clone(clf); m.fit(X_tr,y_train); yp=m.predict(X_te)
        results.append({"Method":nn,"Classifier":cn,"Accuracy":accuracy_score(y_test,yp),"Precision":precision_score(y_test,yp),"Recall":recall_score(y_test,yp),"F1":f1_score(y_test,yp)})
results_df=pd.DataFrame(results)
print(results_df.sort_values("F1",ascending=False).to_string(index=False))

In [ ]:
pivot=results_df.pivot(index="Method",columns="Classifier",values="F1")
fig,ax=plt.subplots(figsize=(9,5))
sns.heatmap(pivot,annot=True,fmt=".4f",cmap="YlGnBu",ax=ax)
ax.set_title("F1: Normalization x Classifier"); plt.tight_layout(); plt.show()

## LazyPredict Benchmark

> Not used for NLP tasks per best practices.

## FLAML AutoML

> Not used for NLP tasks per best practices.

## Additional Best-Library Workflow - spaCy Lemmatization Deep Dive

In [ ]:
nlp_full=spacy.load("en_core_web_sm")
doc=nlp_full("The companies stocks were running better than expected flies")
print(f"{'Token':<15}{'Lemma':<15}{'POS':<8}{'Tag':<12}")
for t in doc: print(f"{t.text:<15}{t.lemma_:<15}{t.pos_:<8}{t.tag_:<12}")
print()
for w in ["companies","stocks","running","better","expected","flies"]:
    d=nlp_full(w)
    print(f"  {w:12s} Porter:{porter.stem(w):12s} Snowball:{snowball.stem(w):12s} Lemma:{d[0].lemma_}")

## Top 3 Model Selection

In [ ]:
top3=results_df.sort_values("F1",ascending=False).head(3).reset_index(drop=True)
print(top3[["Method","Classifier","Accuracy","F1"]].to_string(index=False))
print(f"Baseline F1={bl_f1:.4f}, Improvement: {((top3.iloc[0]['F1']-bl_f1)/bl_f1)*100:+.2f}%")

## Final Training and Evaluation of Top 3

In [ ]:
top3_preds={}
for i,row in top3.iterrows():
    nf=NORMALIZERS[row["Method"]]
    Xtrn=X_train.apply(nf); Xten=X_test.apply(nf)
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2))
    X_tr=tfidf.fit_transform(Xtrn); X_te=tfidf.transform(Xten)
    if row["Classifier"]=="MultinomialNB": m=MultinomialNB()
    elif row["Classifier"]=="LogisticRegression": m=LogisticRegression(max_iter=1000,random_state=SEED)
    else: m=LinearSVC(max_iter=2000,random_state=SEED)
    m.fit(X_tr,y_train); yp=m.predict(X_te)
    top3_preds[f"{row['Method']}+{row['Classifier']}"]=yp
    print(f"\n{'='*60}\n#{i+1}\n{'='*60}")
    print(classification_report(y_test,yp,target_names=CLASS_NAMES))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
for idx,(n,yp) in enumerate(top3_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test,yp),display_labels=CLASS_NAMES).plot(ax=axes[idx],cmap="Blues",values_format="d")
    axes[idx].set_title(f"#{idx+1}",fontsize=9)
plt.tight_layout(); plt.show()

## Error Analysis

In [ ]:
bp=list(top3_preds.values())[0]; em=bp!=y_test.values
edf=pd.DataFrame({"text":X_test.values[em],"true":y_test.values[em],"pred":bp[em]})
print(f"Errors: {len(edf)}/{len(y_test)} ({len(edf)/len(y_test)*100:.1f}%)")

## Interpretation and Business Insight

1. Lemmatization preserves meaning better.
2. Stemming reduces vocab more aggressively.
3. Minimal normalization often works best for sentiment.
4. Classifier matters more than normalization.

## Limitations

Single dataset, English only, TF-IDF only, spaCy is slow.

## How to Improve This Project

1. Multiple languages. 2. WordNet lemmatizer. 3. NER task test. 4. Cross-validation.

## Production Considerations

Speed vs quality trade-off. Consistency. Caching. Language support.

## Common Mistakes

1. Assuming normalization always helps.
2. Porter for non-English.
3. Over-stemming.
4. No POS before lemmatization.
5. Normalizing before tokenization.

## Mini Challenge / Exercises

1. Add WordNetLemmatizer. 2. Test on AG News. 3. Vocab reduction %. 4. Custom stemmer.

## Final Summary / Key Takeaways

1. Always benchmark vs no-normalization.
2. Lemmatization when meaning matters.
3. Stemming when speed matters.
4. Classifier often matters more.
5. Transformers make normalization less critical.